# Python — Shallow Copy vs Deep Copy

Copying objects incorrectly is one of the most common sources of bugs in Python.  
Understanding the difference between **assignment**, **shallow copy**, and **deep copy** is essential.

| Method | What it does | Nested objects |
|--------|-------------|----------------|
| `b = a` | Both variables point to the **same** object | Shared |
| `copy.copy(a)` | New outer object, but inner objects are **shared** | Shared |
| `copy.deepcopy(a)` | Fully independent copy of everything | Independent |

## 1. Assignment — Not a Copy!

`b = a` does **not** create a new object. Both variables point to the same object in memory.  
Changing one changes the other.

In [ ]:
# Assignment — same object, two names
a = [1, 2, 3]
b = a             # NOT a copy — b points to the same list

b.append(4)
print("a after b.append(4):", a)   # [1, 2, 3, 4] — a changed too!
print("b:", b)                     # [1, 2, 3, 4]
print("Same object?", a is b)      # True

# id() shows the memory address — both are the same
print("id(a):", id(a))
print("id(b):", id(b))

## 2. Shallow Copy

Creates a **new outer container** but the elements inside still point to the same objects.  
Safe for flat lists. Dangerous for nested structures.

In [ ]:
import copy

# --- Flat list — shallow copy works fine ---
original = [1, 2, 3, 4]
shallow  = copy.copy(original)   # also: original.copy() or original[:]

shallow.append(99)
print("original:", original)   # [1, 2, 3, 4]   — unaffected
print("shallow :", shallow)    # [1, 2, 3, 4, 99]
print("Same object?", original is shallow)  # False — different containers

print()

# --- Nested list — shallow copy is NOT enough ---
nested   = [[1, 2], [3, 4], [5, 6]]
shallow2 = copy.copy(nested)

shallow2[0].append(99)         # modifying the INNER list
print("nested  :", nested)     # [[1, 2, 99], [3, 4], [5, 6]] — changed!
print("shallow2:", shallow2)   # [[1, 2, 99], [3, 4], [5, 6]]

# Outer containers are different, but inner lists are shared
print("Outer same?    ", nested is shallow2)        # False
print("Inner[0] same?", nested[0] is shallow2[0])  # True — shared!

## 3. Deep Copy

Creates a **completely independent copy** — new outer container AND new copies of all nested objects.  
Changes to the copy never affect the original.

In [ ]:
import copy

nested = [[1, 2], [3, 4], [5, 6]]
deep   = copy.deepcopy(nested)

deep[0].append(99)            # modify the copy's inner list
print("nested:", nested)      # [[1, 2], [3, 4], [5, 6]]  — unchanged!
print("deep  :", deep)        # [[1, 2, 99], [3, 4], [5, 6]]

print("Outer same?    ", nested is deep)        # False
print("Inner[0] same?", nested[0] is deep[0])  # False — fully independent

# Works on complex objects too
data = {
    "team": "Engineering",
    "members": [{"name": "Alice", "skills": ["Python", "SQL"]},
                {"name": "Bob",   "skills": ["Java", "PySpark"]}]
}

data_copy = copy.deepcopy(data)
data_copy["members"][0]["skills"].append("PySpark")

print("\nOriginal skills:", data["members"][0]["skills"])       # ['Python', 'SQL']
print("Copy skills    :", data_copy["members"][0]["skills"])    # ['Python', 'SQL', 'PySpark']

## 4. Copying Shortcuts for Common Types

In [ ]:
# Lists — three ways to shallow copy
lst = [1, 2, 3]
a = lst.copy()      # list method
b = lst[:]          # slice notation
c = list(lst)       # constructor

print("All equal, none the same object:")
print(a == b == c == lst)          # True (equal values)
print(a is lst, b is lst, c is lst) # False False False

# Dicts — shallow copy
d = {"x": 1, "y": [10, 20]}
d_copy = d.copy()           # or dict(d)
d_copy["x"] = 99            # new key — original unaffected
d_copy["y"].append(30)      # modifying nested list — BOTH affected!

print("\nOriginal d:", d)         # {'x': 1, 'y': [10, 20, 30]}
print("Copy d    :", d_copy)      # {'x': 99, 'y': [10, 20, 30]}

# Tuples — tuples are immutable, so 'copying' is usually unnecessary
t = (1, 2, 3)
t2 = t   # same object is fine — tuples can't change
print("\nTuple copy is same object:", t is t2)  # True — safe because immutable

## 5. Visual Summary

```
original = [[1, 2], [3, 4]]

Assignment:    b = original
  original ──► [ ──► [1,2], ──► [3,4] ]
  b        ──┘   (same object entirely)

Shallow copy:  b = copy.copy(original)
  original ──► [ ──► [1,2], ──► [3,4] ]
                        ▲           ▲
  b        ──► [ ───────┘   ────────┘ ]  (new outer, shared inners)

Deep copy:     b = copy.deepcopy(original)
  original ──► [ ──► [1,2], ──► [3,4] ]
  b        ──► [ ──► [1,2], ──► [3,4] ]  (everything new and independent)
```

## 6. When to Use Each

| Situation | Use |
|-----------|-----|
| You want two names for the same object | `b = a` |
| Flat list/dict — no nesting | `copy.copy()` or `.copy()` |
| Nested lists, dicts, or objects | `copy.deepcopy()` |
| DataFrame (PySpark/Pandas) | Neither — DataFrames are immutable by design; transformations return new DFs |

In [ ]:
import copy

# Real-world: saving state before a destructive operation
config = {
    "db": {"host": "localhost", "port": 5432},
    "retries": 3,
    "tags": ["prod", "v2"]
}

# Deep copy before modifying — preserve original
test_config = copy.deepcopy(config)
test_config["db"]["host"] = "test-server"
test_config["tags"].append("test")

print("Original config:", config)
print("Test config    :", test_config)

## Quick Reference

```python
import copy

# Assignment — same object
b = a

# Shallow copy — new container, shared contents
b = copy.copy(a)
b = a.copy()          # list/dict method
b = a[:]              # list slice
b = list(a)           # list constructor
b = dict(a)           # dict constructor

# Deep copy — fully independent
b = copy.deepcopy(a)
```

**Rule of thumb:** If your data has any nesting (list of lists, list of dicts, dicts of dicts), always use `deepcopy`.

## Interview Questions

1. What is the difference between `b = a` and `b = a.copy()`?
2. When does a shallow copy behave the same as a deep copy?
3. What happens when you shallow copy a list that contains other lists?
4. Why don't you need to deep copy a tuple?
5. How does Python's `is` operator differ from `==`?
6. What does `id()` tell you and how does it relate to copying?
7. In PySpark, do you need to worry about shallow vs deep copy? Why or why not?

## Practice Exercises

**Easy:**
1. Create `a = [10, 20, 30]`. Make `b = a`. Append 99 to `b`. Print both — what do you observe?
2. Repeat with `b = a.copy()`. What changes?

**Medium:**
3. Create `data = [[1, 2], [3, 4]]`. Shallow copy it. Modify the inner list via the copy. Show that the original is affected.
4. Repeat with `deepcopy`. Show that the original is now safe.

**Advanced:**
5. Create a class `Config` with a nested `settings` dict. Show what happens when you assign vs shallow copy vs deep copy an instance.
6. Write a function `safe_update(config, updates)` that applies updates to a deep copy and returns the new config without modifying the original.